# 5.0 — Useful Local Agents

## When is a local agent actually useful?

A local agent combines a local LLM with tools, memory, and an execution loop — adding power, and constraints. The honest trade-off:

| Dimension | Local agent | Cloud agent |
|-----------|------------|-------------|
| **Privacy** | Data never leaves your machine | Sent to provider |
| **Cost** | Free after hardware cost | Per-token billing |
| **Latency** | ~1–5 s/step (GPU-dependent) | ~0.5–2 s/step |
| **Context** | 8K–128K (model-dependent) | 128K–1M |
| **Capability** | Strong for bounded tasks | Better for open reasoning |
| **Stack** | You own everything | Provider manages it |

**Rule of thumb (May 2026):**  
Local agents win on *bounded execution* — repeatable, private, or latency-sensitive tasks.  
Cloud models win on *open-ended judgment* — ambiguous planning, long context, high-stakes decisions.

**Productive architecture:** `local model + retrieval + tools + eval set + router/fallback`

In [ ]:
import subprocess, sys


import ollama

def pull_first_available(models):
    for model in models:
        r = subprocess.run(['ollama', 'pull', model], capture_output=True, text=True)
        if r.returncode == 0:
            print(f'Ready: {model}')
            return model
        print(f'  {model}: unavailable, trying next...')
    raise RuntimeError('No model available — install Ollama and pull gemma4, qwen3.6, or llama3.3')

MODEL = pull_first_available(['gemma4', 'qwen3.6', 'llama3.3'])
print(f'Active model: {MODEL}')

## Hermes Agent — Live Demo

[Hermes Agent](https://github.com/NousResearch/hermes-agent) (Nous Research) is a local-first agent product built on top of their Hermes model family. It adds persistent memory, a skill registry, and a TUI — think of it as a self-hosted Cursor for agentic tasks.

It cannot be embedded in a notebook, so this section is a guide for the **live demo** portion of the session.

### Setup (run in terminal before the session)

```bash
curl -fsSL https://raw.githubusercontent.com/NousResearch/hermes-agent/main/scripts/install.sh | bash
hermes model
# → Custom Endpoint → http://localhost:11434 → gemma4 → API key: ollama
```

### Suggested live demo prompts

These three prompts cover easy → medium → hard and highlight what's different from a plain LLM call:

**1. File pipeline (easy) — shows tool dispatch + file creation:**
```
Read README.md from the current directory and save a one-paragraph summary to README_summary.txt.
```

**2. Research + report (medium) — shows search → synthesize → write in one shot:**
```
Search for the top 3 open-source embedding models available today.
Write a comparison table (Model | Dims | License | Use case) to embedding_comparison.md.
```

**3. Persistent memory (hard) — shows what Hermes adds over raw tool calling:**
```
Remember that I prefer markdown output and bullet points over prose.
```
Then in a new session:
```
Summarize what LLM fine-tuning is.
```
Point out that Hermes applied the preference without being told again.

> The notebook demos below implement the same *patterns* using the `ollama` Python library directly — no external product needed, just Ollama running locally.

In [ ]:
from ddgs import DDGS

# --- Real tools ---

def read_file(filename):
    """Read a local file. Caps at 6K chars to stay within model context."""
    try:
        with open(filename, 'r') as f:
            content = f.read()
        if len(content) > 6000:
            content = content[:6000] + '\n[...truncated for context]'
        return content
    except FileNotFoundError:
        return f'File not found: {filename}'

def web_search(query):
    """Search DuckDuckGo and return the top 4 results."""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=4))
    if not results:
        return 'No results found.'
    return '\n\n'.join(
        f"{r['title']}\n{r['href']}\n{r['body']}" for r in results
    )

def write_file(filename, content):
    """Write text to a local file."""
    with open(filename, 'w') as f:
        f.write(content)
    return f'Wrote {len(content)} chars to {filename}'


TOOLS = {'read_file': read_file, 'web_search': web_search, 'write_file': write_file}

TOOLS_SCHEMA = [
    {'type': 'function', 'function': {
        'name': 'read_file',
        'description': 'Read a local file and return its contents',
        'parameters': {
            'type': 'object',
            'properties': {'filename': {'type': 'string', 'description': 'Path to the file'}},
            'required': ['filename']
        }
    }},
    {'type': 'function', 'function': {
        'name': 'web_search',
        'description': 'Search DuckDuckGo for current information',
        'parameters': {
            'type': 'object',
            'properties': {'query': {'type': 'string', 'description': 'Search query'}},
            'required': ['query']
        }
    }},
    {'type': 'function', 'function': {
        'name': 'write_file',
        'description': 'Write text content to a local file',
        'parameters': {
            'type': 'object',
            'properties': {
                'filename': {'type': 'string', 'description': 'Target file path'},
                'content':  {'type': 'string', 'description': 'Text to write'}
            },
            'required': ['filename', 'content']
        }
    }}
]


# --- Agent loop ---

def run_agent(prompt, max_steps=6):
    messages  = [{'role': 'user', 'content': prompt}]
    tool_log  = []

    for step in range(max_steps):
        response = ollama.chat(model=MODEL, messages=messages, tools=TOOLS_SCHEMA)
        msg      = response['message']
        calls    = msg.get('tool_calls', [])

        if not calls:
            print(f'\nFinal answer:\n{msg["content"]}')
            return {'response': msg['content'], 'tool_log': tool_log, 'steps': step + 1}

        messages.append(msg)

        for tc in calls:
            fn   = tc['function']['name']
            args = tc['function']['arguments']
            arg_preview = ', '.join(f'{k}={repr(v)[:50]}' for k, v in args.items())
            print(f'  [{step + 1}] {fn}({arg_preview})')

            out = TOOLS[fn](**args) if fn in TOOLS else f'Unknown tool: {fn}'
            tool_log.append({'step': step + 1, 'tool': fn, 'args': args})
            messages.append({'role': 'tool', 'content': str(out)})

    return {'response': None, 'tool_log': tool_log, 'steps': max_steps}


print('Tools ready:', list(TOOLS.keys()))

## Demo A — Private Document Analyst  *(Easy)*

**Difficulty:** Single tool chain, local file only, deterministic output.

**Scenario:** You have a technical paper locally — a draft, a client report, internal research. You need to extract key information from it. With a local agent, the content never reaches an external server.

The agent will:
1. Read `text_files/lora-paper.txt` (the LoRA fine-tuning paper)
2. Extract the 3 main technical contributions
3. Write a structured summary to `lora_summary.txt`

After it runs, open `lora_summary.txt` and judge the output yourself.

In [ ]:
print(f'Model: {MODEL}')
print('Task: Read LoRA paper → extract 3 key contributions → write to lora_summary.txt')
print()

TASK_A = (
    "Read the file 'text_files/lora-paper.txt'. "
    "Identify the 3 main technical contributions of the LoRA paper "
    "(Low-Rank Adaptation of Large Language Models). "
    "Keep each contribution to a maximum of 60 words — be precise, not vague. "
    "Write the result to 'lora_summary.txt' using this format:\n"
    "# LoRA — Key Contributions\n"
    "1. [contribution]\n"
    "2. [contribution]\n"
    "3. [contribution]"
)

result_a = run_agent(TASK_A)

# ── GPT-4o baseline (uncomment to compare) ───────────────────────────────────
# Swap MODEL for any OpenAI-compatible endpoint (LiteLLM, OpenRouter, etc.)
# _local = MODEL; MODEL = "gpt-4o"
# result_a_baseline = run_agent(TASK_A)
# MODEL = _local

## Demo B — Real-time Research Pipeline  *(Medium)*

**Difficulty:** Live web search + synthesis + file write. Results vary with what the internet returns.

**Scenario:** You need a quick briefing on what's happening in the local-AI space. Instead of browsing manually, you want the agent to search, synthesize, and produce a structured output — all locally, for free.

The agent will:
1. Run a real DuckDuckGo search for recent open-source LLM releases
2. Pick the 3 most interesting results
3. Write a short markdown briefing to `recent_models.md`

The search results are live — you'll see what the model retrieves and how it synthesizes them. If the first search isn't enough, watch whether the agent runs a follow-up.

In [ ]:
from datetime import datetime

_now        = datetime.now()
_month_year = _now.strftime("%B %Y")   # e.g. "May 2026"
_year       = _now.strftime("%Y")       # e.g. "2026"

print(f'Model: {MODEL}')
print(f'Task: Search for recent open-source LLM releases ({_month_year}) → write markdown briefing')
print()

TASK_B = (
    f"Search for 'open source LLM model releases {_month_year}'. "
    f"If results are sparse, also search for 'new local LLM {_year} site:huggingface.co OR site:ollama.com'. "
    "From what you find, pick the 3 most interesting recent model releases. "
    "Write a markdown file 'recent_models.md' with a short paragraph on each: "
    "what the model is, who made it, and why it matters for running locally. "
    "If the first search doesn't give enough detail on a specific model, run a targeted follow-up search on it. "
    "Write whatever you found — do not say you cannot complete the task."
)

result_b = run_agent(TASK_B)

# ── GPT-4o baseline (uncomment to compare) ───────────────────────────────────
# _local = MODEL; MODEL = "gpt-4o"
# result_b_baseline = run_agent(TASK_B)
# MODEL = _local

## Demo C — Cross-Source Synthesis  *(Hard)*

**Difficulty:** Multi-hop reasoning across a local file and live web. Requires chaining tools and synthesising two different knowledge sources into a structured comparison.

**Scenario:** You've just extracted the key contributions of the LoRA paper. Now you want to know how the field has moved on — what's been improved, replaced, or built on top of LoRA since it was published. The agent must combine what it read locally with what it finds on the web.

The agent will:
1. Read `lora_summary.txt` (the output from Demo A)
2. Search for recent LoRA improvements and alternatives (adapters, QLoRA, DoRA, etc.)
3. Write `lora_evolution.md` — a before/after comparison of the original approach vs. current state-of-the-art

This is the hardest task: it requires the agent to reason across two different information sources and produce a structured synthesis, not just a summary.

After it runs, check whether `lora_evolution.md` actually compares the two sources or just repeats one of them.

In [ ]:
from datetime import datetime as _dt
_year = _dt.now().strftime("%Y")

print(f'Model: {MODEL}')
print('Task: Read LoRA summary → search for recent alternatives → write before/after comparison')
print()

TASK_C = (
    "First, read the file 'lora_summary.txt' to understand the original LoRA contributions. "
    f"Then search for 'LoRA improvements alternatives {_year} QLoRA DoRA adapter fine-tuning'. "
    "If you need more detail on a specific technique, run a follow-up search on it. "
    "Write a file 'lora_evolution.md' with two sections:\n"
    "## Original LoRA (from paper)\n"
    "[3 bullet points from lora_summary.txt]\n\n"
    "## Where the Field Has Gone\n"
    "[3 bullet points on what has improved, replaced, or built on top of LoRA — cite technique names]\n\n"
    "## Verdict\n"
    "[1 paragraph: is vanilla LoRA still the right choice for fine-tuning, or has something better taken over?]"
)

result_c = run_agent(TASK_C)

# ── GPT-4o baseline (uncomment to compare) ───────────────────────────────────
# This is where the gap between local and cloud models is most visible.
# _local = MODEL; MODEL = "gpt-4o"
# result_c_baseline = run_agent(TASK_C)
# MODEL = _local

## Vibe-Check — Rate the Outputs Yourself

Automated scoring by keyword matching or response length is not evaluation. The only honest check is a human reading the output.

The next cell will ask you to score each demo on three dimensions:

| Dimension | What to look at |
|-----------|----------------|
| **Finished?** | Did it complete the stated goal — file created, table written, question answered? |
| **Tool calls correct (1–5)** | Right tool, right order, right arguments? Did it waste steps? |
| **Answer quality (1–5)** | Is the output you'd actually use? Accurate, specific, well-structured? |

Before scoring, open the output files and read them:
- **Demo A (Easy):** `lora_summary.txt`
- **Demo B (Medium):** `recent_models.md`
- **Demo C (Hard):** `lora_evolution.md`

---

> **Note — this is a 3-task walkthrough, not a production eval.**
>
> Three tasks across Easy / Medium / Hard give you a useful signal about where the model breaks down, but they are not enough to trust it in production. A real vibe-check evaluation uses **20–30 diverse examples** that cover:
> - **Happy path** — clean inputs, straightforward tasks
> - **Edge cases** — ambiguous queries, missing files, empty search results
> - **Failure modes** — tasks the model is likely to hallucinate or abandon
> - **Consistency** — running the same prompt multiple times to check variance
>
> Only once you've scored across that range do the averages mean something. Tools like [RAGAS](https://github.com/explodinggradients/ragas), [promptfoo](https://github.com/promptfoo/promptfoo), or a simple spreadsheet work well for this. The scoring dimensions above transfer directly.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

print('=' * 55)
print('VIBE CHECK — Rate the outputs you just saw')
print('=' * 55)

def make_demo_widgets(label):
    finish  = widgets.Dropdown(
        options=[('No (0)', 0), ('Partial (1)', 1), ('Yes (2)', 2)],
        description='Finished?', style={'description_width': '80px'})
    tools   = widgets.IntSlider(min=1, max=5, value=3, description='Tools/5:',
        style={'description_width': '80px'})
    quality = widgets.IntSlider(min=1, max=5, value=3, description='Quality/5:',
        style={'description_width': '80px'})
    box = widgets.VBox([widgets.HTML(f'<b>{label}</b>'), finish, tools, quality])
    return box, finish, tools, quality

box_a, a_finish, a_tools, a_quality = make_demo_widgets('Demo A (Easy) — lora_summary.txt')
box_b, b_finish, b_tools, b_quality = make_demo_widgets('Demo B (Medium) — recent_models.md')
box_c, c_finish, c_tools, c_quality = make_demo_widgets('Demo C (Hard) — lora_evolution.md')

btn = widgets.Button(description='Show Results', button_style='success')
out = widgets.Output()

def show_results(_):
    out.clear_output(wait=True)
    with out:
        scores = {
            'Demo A (Easy)':   {'finish': a_finish.value, 'tools': a_tools.value, 'quality': a_quality.value},
            'Demo B (Medium)': {'finish': b_finish.value, 'tools': b_tools.value, 'quality': b_quality.value},
            'Demo C (Hard)':   {'finish': c_finish.value, 'tools': c_tools.value, 'quality': c_quality.value},
        }
        header = f"{'Demo':<20} {'Finish':>8} {'Tools/5':>8} {'Quality/5':>10}"
        print()
        print(header)
        print('-' * len(header))
        for name, s in scores.items():
            print(f"{name:<20} {s['finish']:>8} {s['tools']:>8} {s['quality']:>10}")
        avg_t = sum(s['tools']   for s in scores.values()) / len(scores)
        avg_q = sum(s['quality'] for s in scores.values()) / len(scores)
        print(f'\nAvg tool score: {avg_t:.1f}/5   Avg quality: {avg_q:.1f}/5')
        print()
        if avg_q >= 4 and avg_t >= 4:
            print('→ Strong across all difficulties — local model is viable for these task types.')
        elif scores['Demo C (Hard)']['quality'] <= 2:
            print('→ Model handles easy/medium well but struggles with multi-hop synthesis. Consider a cloud fallback for hard tasks.')
        elif avg_t < 3 or avg_q < 3:
            print('→ These task types may need a stronger model or a cloud fallback.')
        else:
            print('→ Local agent is handling most task types well.')

btn.on_click(show_results)
display(widgets.VBox([box_a, box_b, box_c, btn, out]))

## When NOT to Use a Local Agent

Local agents should route to cloud models when the task involves:

1. **Long-context synthesis** — connecting details across many long documents (>64K tokens in practice)
2. **Multi-hop reasoning** — logic chains where one wrong step breaks the whole answer
3. **Code-heavy edits beyond ~200 LOC** — local models lose coherence over large codebases
4. **Ambiguous planning** — inferring goals, constraints, and trade-offs without explicit guidance
5. **High-stakes decisions** — medical, legal, financial, or irreversible operational actions

**A simple routing rule:**

```python
def should_use_local_agent(task):
    return (
        task.is_bounded          # clear input/output contract
        and task.is_reversible   # tool calls can be undone if wrong
        and task.fits_context    # comfortably under 64K tokens
        and not task.is_high_stakes
    )
```

**On model choice:**
- Gemma 4 27B: best reasoning quality for local agents as of May 2026
- Gemma 4 E4B (4B): faster and lighter for simple, latency-sensitive tasks
- Qwen 3.6: strong tool-call accuracy, good fallback if gemma4 is unavailable

---

*References: [Hermes Agent](https://github.com/NousResearch/hermes-agent) · [Local LLM Usability Guide — May 2026](https://www.notion.so/Local-LLM-Usability-Guide-Updated-May-2026-35aaf36947a180299860fdafa06b3b99)*